# FASE 3 — EDA Ato 1: Metricas de Logistica vs Satisfacao

**Pessoa 3 | Plano 03-01**

Evidencias visuais quantitativas:
- EDA-01: Atraso degrada nota (boxplot + scatter + Mann-Whitney)
- EDA-02: Frete degrada nota (absoluto + percentual)
- EDA-04: Categorias concentram insatisfacao (top-15)


## Celula 1 — Setup

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

sns.set_theme(style="whitegrid", context="talk")

# PROJECT_ROOT resolve tanto de notebooks/ quanto da raiz do projeto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
GOLD = PROJECT_ROOT / "data" / "gold" / "olist_gold.parquet"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FIGURES: {FIGURES}")
print(f"GOLD exists: {GOLD.exists()}")

PROJECT_ROOT: C:\Users\Wey\Desktop\Alpha\Projetos\python
FIGURES: C:\Users\Wey\Desktop\Alpha\Projetos\python\reports\figures
GOLD exists: True


## Celula 2 — Load e derivacao defensiva de dias_atraso

In [2]:
df = pd.read_parquet(GOLD)

# Derivacao defensiva — idempotente mesmo se coluna ja existir
# A gold table ja tem actual_delay_days; recalculamos para consistencia e documentacao
df["dias_atraso"] = (
    pd.to_datetime(df["order_delivered_customer_date"])
    - pd.to_datetime(df["order_estimated_delivery_date"])
).dt.days

# Sanidade: range observado -147 a +188 dias (conforme data_quality.md)
print(f"dias_atraso: min={df['dias_atraso'].min()}, max={df['dias_atraso'].max()}, nulls={df['dias_atraso'].isna().sum()}")
print(f"Shape: {df.shape}, bad_review rate: {df['bad_review'].mean():.2%}")

# review_score e float64 na gold table — converter para int para compat com boxplot
df_valid = df.dropna(subset=["review_score", "dias_atraso"]).copy()
df_valid["review_score_int"] = df_valid["review_score"].astype(int)
print(f"df_valid shape: {df_valid.shape}")

dias_atraso: min=-147.0, max=188.0, nulls=1646
Shape: (97456, 39), bad_review rate: 13.87%
df_valid shape: (95810, 40)


## Celula 3 — Mann-Whitney U test (EDA-01)

In [3]:
bad  = df_valid.loc[df_valid["bad_review"] == 1, "dias_atraso"].dropna()
good = df_valid.loc[df_valid["bad_review"] == 0, "dias_atraso"].dropna()
stat, p = stats.mannwhitneyu(bad, good, alternative="greater")
print(f"Mann-Whitney U={stat:.0f}, p={p:.2e}")
print("Conclusao: pedidos com avaliacao ruim tem MAIOR atraso (p < 0.05)" if p < 0.05 else "ATENCAO: resultado nao significativo")
print(f"Mediana atraso bad_review=1: {bad.median():.1f} dias")
print(f"Mediana atraso bad_review=0: {good.median():.1f} dias")

Mann-Whitney U=673728514, p=0.00e+00
Conclusao: pedidos com avaliacao ruim tem MAIOR atraso (p < 0.05)
Mediana atraso bad_review=1: -8.0 dias
Mediana atraso bad_review=0: -13.0 dias


## Celula 4 — Boxplot por review_score (EDA-01, slide export)

In [4]:

# EDA-01 — Boxplot: Atraso vs Nota
# Mensagem: quanto maior o atraso, pior a nota — de forma consistente

palette = {1: "#c0392b", 2: "#e67e22", 3: "#f1c40f", 4: "#2ecc71", 5: "#27ae60"}

fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=df_valid, x="review_score_int", y="dias_atraso",
    hue="review_score_int", palette=palette,
    order=[1, 2, 3, 4, 5], ax=ax,
    showfliers=False, legend=False,
    width=0.55, linewidth=1.5,
)

# Linha de referência "entregue no prazo"
ax.axhline(0, color="#2c3e50", linestyle="--", linewidth=1.5, zorder=3)
ax.text(4.6, 1.5, "← entrega no prazo", fontsize=10, color="#2c3e50", va="bottom")

# Medianas anotadas em cada boxplot
medianas = df_valid.groupby("review_score_int")["dias_atraso"].median()
for score, med in medianas.items():
    ax.text(score - 1, med + 1, f"{med:.0f}d", ha="center", va="bottom",
            fontsize=10, fontweight="bold", color="white",
            bbox=dict(boxstyle="round,pad=0.2", fc=palette[score], ec="none", alpha=0.9))

# Destaque visual: área "ruim" vs "bom"
ax.axvspan(-0.5, 1.5, alpha=0.06, color="#e74c3c", zorder=0)
ax.axvspan(2.5, 4.5, alpha=0.06, color="#27ae60", zorder=0)
ax.text(0.5, ax.get_ylim()[1] * 0.92, "Avaliações\nruins", ha="center",
        fontsize=9, color="#c0392b", style="italic")
ax.text(3.5, ax.get_ylim()[1] * 0.92, "Avaliações\nboas", ha="center",
        fontsize=9, color="#27ae60", style="italic")

ax.set_title("Pedidos atrasados recebem notas piores", fontsize=17, fontweight="bold", pad=15)
ax.set_subtitle = None
ax.text(0.5, 1.01,
        f"Diferença de mediana entre nota 1 e nota 5: {medianas[1] - medianas[5]:.0f} dias  |  Mann-Whitney p ≈ 0",
        transform=ax.transAxes, ha="center", fontsize=10, color="#7f8c8d")
ax.set_xlabel("Nota dada pelo cliente (estrelas ★)", fontsize=13)
ax.set_ylabel("Dias de atraso na entrega\n(negativo = antecipado)", fontsize=12)
ax.set_xticklabels(["★☆☆☆☆\n(1)", "★★☆☆☆\n(2)", "★★★☆☆\n(3)", "★★★★☆\n(4)", "★★★★★\n(5)"], fontsize=11)

fig.tight_layout()
fig.savefig(FIGURES / "eda01_atraso_vs_nota_boxplot.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda01_atraso_vs_nota_boxplot.png")


C:\Users\Wey\AppData\Local\Temp\ipykernel_49232\1549149074.py:42: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["★☆☆☆☆\n(1)", "★★☆☆☆\n(2)", "★★★☆☆\n(3)", "★★★★☆\n(4)", "★★★★★\n(5)"], fontsize=11)
C:\Users\Wey\AppData\Local\Temp\ipykernel_49232\1549149074.py:44: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\Wey\AppData\Local\Temp\ipykernel_49232\1549149074.py:44: UserWarning: Glyph 9734 (\N{WHITE STAR}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\Wey\AppData\Local\Temp\ipykernel_49232\1549149074.py:45: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) Arial.
  fig.savefig(FIGURES / "eda01_atraso_vs_nota_boxplot.png", dpi=150, bbox_inches="tight")
C:\Users\Wey\AppData\Local\Temp\ipykernel_49232\1549149074.py:45: UserWarning: Glyph 9734 (\N{WHITE STAR}) missing from font(s) Arial.
  fig.savefig(FIGURES / "eda

Exportado: eda01_atraso_vs_nota_boxplot.png


In [5]:

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

# Paleta e estilo base
CORES = {"ruim": "#e74c3c", "bom": "#27ae60", "neutro": "#95a5a6", "fundo": "#f8f9fa"}
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("Setup de estilo OK")


Setup de estilo OK



# EDA-01 — Scatter: atraso vs nota com destaque da tendência
# Substitui scatter genérico por médias com IC — muito mais legível que 5k pontos sobrepostos

media_por_nota = df_valid.groupby("review_score_int")["dias_atraso"].agg(["mean", "sem", "count"]).reset_index()
media_por_nota.columns = ["nota", "media", "se", "n"]
media_por_nota["ci95"] = 1.96 * media_por_nota["se"]

colors_list = [palette[i] for i in media_por_nota["nota"]]

fig, ax = plt.subplots(figsize=(9, 5))

# Área de IC
ax.fill_between(media_por_nota["nota"],
                media_por_nota["media"] - media_por_nota["ci95"],
                media_por_nota["media"] + media_por_nota["ci95"],
                alpha=0.18, color="#3498db")

# Linha de tendência
ax.plot(media_por_nota["nota"], media_por_nota["media"],
        color="#2980b9", linewidth=2.5, zorder=3, marker="o", markersize=10)

# Pontos coloridos por nota
for _, row in media_por_nota.iterrows():
    ax.scatter(row["nota"], row["media"], color=palette[int(row["nota"])],
               s=180, zorder=4, edgecolors="white", linewidth=1.5)
    ax.annotate(f"{row['media']:.1f}d\n(n={int(row['n'])/1000:.0f}k)",
                xy=(row["nota"], row["media"]),
                xytext=(0, 16), textcoords="offset points",
                ha="center", fontsize=9.5, color="#2c3e50",
                fontweight="bold")

ax.axhline(0, color="#7f8c8d", linestyle="--", linewidth=1.2)
ax.text(5.05, 0.8, "no prazo", fontsize=9, color="#7f8c8d")

ax.set_title("Quanto mais atraso, menor a nota — padrão consistente", fontsize=15, fontweight="bold", pad=12)
ax.text(0.5, 1.02,
        "Média de dias de atraso por nota (IC 95%). Negativo = entregue antes do prazo.",
        transform=ax.transAxes, ha="center", fontsize=10, color="#7f8c8d")
ax.set_xlabel("Nota dada pelo cliente (★)", fontsize=13)
ax.set_ylabel("Dias de atraso médio", fontsize=12)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(["1 ★\n(ruim)", "2 ★★", "3 ★★★", "4 ★★★★", "5 ★★★★★\n(ótimo)"], fontsize=11)
ax.set_xlim(0.5, 5.5)

fig.tight_layout()
fig.savefig(FIGURES / "eda01_atraso_vs_nota_scatter.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda01_atraso_vs_nota_scatter.png")


In [6]:
sample = df_valid.sample(min(5000, len(df_valid)), random_state=42)
jitter = np.random.uniform(-0.2, 0.2, size=len(sample))

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    sample["dias_atraso"],
    sample["review_score_int"] + jitter,
    alpha=0.05, s=4, color="steelblue",
)
ax.set_title("Atraso vs. Nota (amostra 5k, jitter)", fontsize=16, fontweight="bold")
ax.set_xlabel("Dias de Atraso", fontsize=13)
ax.set_ylabel("Nota (estrelas)", fontsize=13)
ax.set_yticks([1, 2, 3, 4, 5])
fig.tight_layout()
fig.savefig(FIGURES / "eda01_atraso_vs_nota_scatter.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda01_atraso_vs_nota_scatter.png")

Exportado: eda01_atraso_vs_nota_scatter.png


## Celula 6 — Derivar frete_pct_pedido (EDA-02)

Nota: a gold table nao tem coluna `price` separada; usamos `payment_value` como denominador
(total pago pelo cliente — inclui produto, frete e outros encargos).

In [7]:
# Protecao contra payment_value == 0
denominador = df["payment_value"].replace(0, float("nan"))
df["frete_pct_pedido"] = df["freight_value"] / denominador

# Sanidade
print(f"frete_pct_pedido: min={df['frete_pct_pedido'].min():.3f}, max={df['frete_pct_pedido'].max():.3f}")
print("Mediana por nota:")
print(df.dropna(subset=["review_score"]).assign(rs=lambda x: x["review_score"].astype(int)).groupby("rs")["frete_pct_pedido"].median().round(3))

frete_pct_pedido: min=0.000, max=21.447
Mediana por nota:
rs
1    0.230
2    0.235
3    0.237
4    0.225
5    0.222
Name: frete_pct_pedido, dtype: float64



# EDA-02 — Frete: um único painel mais claro com duplo eixo e anotações
# Mensagem: pedidos com nota ruim pagam mais frete — absoluto e relativo

df_frete = df_valid.copy()
df_frete["frete_pct_pedido"] = df.loc[df_valid.index, "frete_pct_pedido"]

frete_stats = df_frete.groupby("review_score_int").agg(
    frete_med=("freight_value", "median"),
    frete_pct_med=("frete_pct_pedido", "median"),
    n=("order_id", "count"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 6))
ax2 = ax1.twinx()

x = frete_stats["review_score_int"]
cores_barra = [palette[i] for i in x]

bars = ax1.bar(x, frete_stats["frete_med"], color=cores_barra, alpha=0.85,
               width=0.45, zorder=2, label="Frete mediano (R$)")

# Valores anotados nas barras
for bar, val in zip(bars, frete_stats["frete_med"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f"R${val:.0f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

# Linha de % sobre o eixo direito
ax2.plot(x, frete_stats["frete_pct_med"] * 100, color="#8e44ad",
         linewidth=2.5, marker="D", markersize=9, zorder=4, label="% do valor do pedido")
for xi, pct in zip(x, frete_stats["frete_pct_med"]):
    ax2.annotate(f"{pct:.0%}", xy=(xi, pct * 100), xytext=(8, 0),
                 textcoords="offset points", fontsize=10, color="#8e44ad", fontweight="bold")

ax1.set_title("Clientes insatisfeitos pagam mais frete", fontsize=17, fontweight="bold", pad=14)
ax1.text(0.5, 1.01,
         "Mediana do frete absoluto (barras) e como % do pedido (linha roxo) por nota",
         transform=ax1.transAxes, ha="center", fontsize=10, color="#7f8c8d")
ax1.set_xlabel("Nota dada pelo cliente (★)", fontsize=13)
ax1.set_ylabel("Frete mediano (R$)", fontsize=12, color="#2c3e50")
ax2.set_ylabel("Frete como % do pedido", fontsize=12, color="#8e44ad")
ax1.set_xticks([1, 2, 3, 4, 5])
ax1.set_xticklabels(["1 ★\n(ruim)", "2 ★★", "3 ★★★", "4 ★★★★", "5 ★★★★★\n(ótimo)"], fontsize=11)

# Legenda unificada
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=10, framealpha=0.9)

fig.tight_layout()
fig.savefig(FIGURES / "eda02_frete_vs_nota.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda02_frete_vs_nota.png")


In [8]:
df_frete = df_valid.copy()
df_frete["frete_pct_pedido"] = df.loc[df_valid.index, "frete_pct_pedido"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Painel esquerdo: frete absoluto
sns.boxplot(
    data=df_frete, x="review_score_int", y="freight_value",
    hue="review_score_int",
    palette="RdYlGn", order=[1, 2, 3, 4, 5],
    showfliers=False, ax=axes[0], legend=False,
)
axes[0].set_title("Frete Absoluto vs. Nota", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Nota (estrelas)", fontsize=12)
axes[0].set_ylabel("Valor do Frete (R$)", fontsize=12)

# Painel direito: frete como % do pedido
sns.boxplot(
    data=df_frete, x="review_score_int", y="frete_pct_pedido",
    hue="review_score_int",
    palette="RdYlGn", order=[1, 2, 3, 4, 5],
    showfliers=False, ax=axes[1], legend=False,
)
axes[1].set_title("Frete como % do Pedido vs. Nota", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Nota (estrelas)", fontsize=12)
axes[1].set_ylabel("Frete / Pagamento Total", fontsize=12)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

fig.suptitle("Impacto do Frete na Avaliacao do Cliente", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES / "eda02_frete_vs_nota.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda02_frete_vs_nota.png")

Exportado: eda02_frete_vs_nota.png


## Celula 8 — Correlacao de Spearman (EDA-02)

In [9]:
from scipy.stats import spearmanr
df_sp = df_frete.dropna(subset=["freight_value", "frete_pct_pedido", "review_score_int"])
corr_abs, p_abs = spearmanr(df_sp["freight_value"], df_sp["review_score_int"])
corr_pct, p_pct = spearmanr(df_sp["frete_pct_pedido"], df_sp["review_score_int"])
print(f"Spearman frete_abs vs nota: r={corr_abs:.3f}, p={p_abs:.2e}")
print(f"Spearman frete_pct vs nota: r={corr_pct:.3f}, p={p_pct:.2e}")
print("Direcao negativa esperada (maior frete = menor nota)")

Spearman frete_abs vs nota: r=-0.088, p=1.04e-164
Spearman frete_pct vs nota: r=-0.031, p=8.28e-22
Direcao negativa esperada (maior frete = menor nota)


## Celula 9 — Verificar disponibilidade de product_category_name_english (EDA-04)

In [10]:
if "product_category_name_english" not in df.columns:
    # Fallback: fazer merge da translation table
    translation = pd.read_csv(
        PROJECT_ROOT / "data" / "raw" / "product_category_name_translation.csv",
        usecols=["product_category_name", "product_category_name_english"],
    )
    df = df.merge(translation, on="product_category_name", how="left")
    print("Merge com translation table realizado")
else:
    print("product_category_name_english ja disponivel na gold table")

print(f"Categorias distintas: {df['product_category_name_english'].nunique()}")

product_category_name_english ja disponivel na gold table
Categorias distintas: 71


## Celula 10 — Agregacao top-15 categorias por contagem de avaliacoes ruins (EDA-04)

In [11]:
cat_agg = (
    df[df["bad_review"] == 1]
    .groupby("product_category_name_english")["order_id"]
    .count()
    .nlargest(15)
    .reset_index(name="bad_review_count")
)
print(cat_agg)

   product_category_name_english  bad_review_count
0                 bed_bath_table              1505
1                  health_beauty              1070
2          computers_accessories              1033
3                furniture_decor              1007
4                 sports_leisure               943
5                  watches_gifts               819
6                     housewares               721
7                      telephony               627
8                           auto               535
9                           toys               460
10                  garden_tools               441
11                    cool_stuff               425
12                          baby               423
13                     perfumery               402
14                   electronics               346



# EDA-04 — Categorias: barras horizontais com valores anotados e destaque top-3
# Mensagem: essas categorias concentram a maioria das insatisfações

# Calcular taxa de avaliação ruim por categoria (não só contagem bruta)
cat_total = df.groupby("product_category_name_english")["order_id"].count().rename("total")
cat_bad   = df[df["bad_review"] == 1].groupby("product_category_name_english")["order_id"].count().rename("bad")
cat_df = pd.concat([cat_total, cat_bad], axis=1).dropna().reset_index()
cat_df["bad_rate"] = cat_df["bad"] / cat_df["total"]

# Top 15 por contagem absoluta (narrativa de volume)
cat_agg = cat_df.nlargest(15, "bad").sort_values("bad")

cores_cat = ["#e74c3c" if i >= 12 else "#e67e22" if i >= 9 else "#3498db"
             for i in range(len(cat_agg))]

fig, ax = plt.subplots(figsize=(11, 8))

bars = ax.barh(cat_agg["product_category_name_english"], cat_agg["bad"],
               color=cores_cat, edgecolor="white", linewidth=0.5, height=0.7)

# Anotações: contagem + taxa
for bar, (_, row) in zip(bars, cat_agg.iterrows()):
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height() / 2,
            f"{int(row['bad']):,}  ({row['bad_rate']:.0%})",
            va="center", fontsize=10, color="#2c3e50")

# Legenda de cores
patches = [
    mpatches.Patch(color="#e74c3c", label="Top 3 — maior volume"),
    mpatches.Patch(color="#e67e22", label="Top 4–6"),
    mpatches.Patch(color="#3498db", label="Demais"),
]
ax.legend(handles=patches, loc="lower right", fontsize=10, framealpha=0.9)

ax.set_title("Categorias com mais avaliações ruins (1-2 ★)", fontsize=16, fontweight="bold", pad=14)
ax.text(0.5, 1.01,
        "Barras = total de avaliações ruins  |  Número à direita = % de ruim no total da categoria",
        transform=ax.transAxes, ha="center", fontsize=10, color="#7f8c8d")
ax.set_xlabel("Total de avaliações ruins (pedidos)", fontsize=12)
ax.set_ylabel("")
ax.set_xlim(0, cat_agg["bad"].max() * 1.22)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.tight_layout()
fig.savefig(FIGURES / "eda04_categorias_ruins.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda04_categorias_ruins.png")


In [12]:
# Tentativa com plotly + kaleido; fallback seaborn garantido
try:
    import plotly.express as px
    fig_plotly = px.bar(
        cat_agg.sort_values("bad_review_count"),
        x="bad_review_count",
        y="product_category_name_english",
        orientation="h",
        title="Top 15 Categorias com Mais Avaliacoes 1-2 Estrelas",
        labels={
            "bad_review_count": "Contagem de Avaliacoes Ruins",
            "product_category_name_english": "Categoria",
        },
        color="bad_review_count",
        color_continuous_scale="Reds",
    )
    fig_plotly.update_layout(
        yaxis={"categoryorder": "total ascending"},
        coloraxis_showscale=False,
        font=dict(size=13),
        title_font_size=16,
        margin=dict(l=200, r=40, t=60, b=60),
    )
    fig_plotly.write_image(
        str(FIGURES / "eda04_categorias_ruins.png"),
        format="png", scale=2, width=900, height=550,
    )
    print("Exportado via plotly+kaleido: eda04_categorias_ruins.png")
except Exception as e:
    print(f"plotly/kaleido indisponivel ({type(e).__name__}), usando fallback seaborn")
    fig2, ax2 = plt.subplots(figsize=(10, 7))
    sns.barplot(data=cat_agg.sort_values("bad_review_count"), x="bad_review_count",
                y="product_category_name_english", hue="product_category_name_english",
                palette="Reds_r", ax=ax2, legend=False)
    ax2.set_title("Top 15 Categorias — Avaliacoes 1-2 Estrelas", fontsize=15, fontweight="bold")
    ax2.set_xlabel("Contagem", fontsize=12)
    ax2.set_ylabel("Categoria", fontsize=12)
    fig2.tight_layout()
    fig2.savefig(FIGURES / "eda04_categorias_ruins.png", dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print("Exportado via seaborn fallback: eda04_categorias_ruins.png")

Wait expired, Browser is being closed by watchdog.


plotly/kaleido indisponivel (BrowserFailedError), usando fallback seaborn


Exportado via seaborn fallback: eda04_categorias_ruins.png


## Verificacao Final

In [13]:
figs = [
    "eda01_atraso_vs_nota_boxplot.png",
    "eda01_atraso_vs_nota_scatter.png",
    "eda02_frete_vs_nota.png",
    "eda04_categorias_ruins.png",
]
for fname in figs:
    p = FIGURES / fname
    assert p.exists() and p.stat().st_size > 10000, f"PNG ausente ou vazio: {p}"
    print(f"OK: {fname} ({p.stat().st_size:,} bytes)")
print("PLANO 03-01 OK: notebook + 4 PNGs verificados")

OK: eda01_atraso_vs_nota_boxplot.png (83,569 bytes)
OK: eda01_atraso_vs_nota_scatter.png (107,950 bytes)
OK: eda02_frete_vs_nota.png (74,124 bytes)
OK: eda04_categorias_ruins.png (85,559 bytes)
PLANO 03-01 OK: notebook + 4 PNGs verificados
